# Crypto.Net · 06 — Cosmos SDK

*Built by Gravicode Studios, led by Kang Fadhil.*

> **Safe to run.** This notebook only reads public networks and signs offline; it never broadcasts a transaction. Keys created here are demo keys — never send real funds to them.

Open in VS Code with the **Polyglot Notebooks** extension (or Jupyter with .NET Interactive) and run the cells in order. [Versi Bahasa Indonesia](./06-cosmos.id.ipynb)

Bech32 accounts across Cosmos chains, live REST reads, protobuf transactions in `SIGN_MODE_DIRECT` and a gas simulation by a real node.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.0.0"

In [ ]:
using Crypto.Net.Core;
using Crypto.Net.Native;
using Crypto.Net.Wallet;
using Crypto.Net.Cosmos;

// The public BIP-39 test phrase has an account on Cosmos Hub, which lets us run a real simulation below.
const string TestPhrase = "abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about";
var wallet  = HdWallet.FromMnemonic(TestPhrase);
var account = wallet.GetCosmosAccount(0, CosmosChain.CosmosHub);   // m/44'/118'/0'/0/0, like Keplr
account.Address.Value

## One key, many chains

The same key has an address on every Cosmos chain that uses coin type 118; only the Bech32 prefix changes.

In [ ]:
new[] { "cosmos", "osmo", "celestia", "juno" }
    .Select(hrp => new { Prefix = hrp, Address = CosmosAddress.ConvertPrefix(account.Address.Value, hrp) })

## Live data

In [ ]:
var rest = new CosmosRestClient(CosmosChain.CosmosHub);
var (accountNumber, sequence) = await rest.GetAccountAsync(account.Address.Value);
Console.WriteLine($"Cosmos Hub height {await rest.GetBlockNumberAsync():N0}");
Console.WriteLine($"account number {accountNumber}, sequence {sequence}");
Console.WriteLine($"balance {await rest.GetBalanceAsync(account.Address)} ATOM");

## Build and sign (SIGN_MODE_DIRECT)

A bank send and a delegation in one transaction. The signature covers SHA-256 of the protobuf `SignDoc` (body, auth info, chain id, account number).

In [ ]:
var builder = new CosmosTxBuilder { Memo = "Crypto.Net notebook", GasLimit = 250_000 }
    .Add(CosmosMessage.BankSend(account.Address.Value, account.Address.Value, new Coin("uatom", 1)))
    .Add(CosmosMessage.Delegate(account.Address.Value, "cosmosvaloper1tflk30mq5vgqjdly92kkhhq3raev2hnz6eete3", new Coin("uatom", 1)));
builder.Fee.Add(new Coin("uatom", 6_250));

byte[] txRaw = account.Sign(builder, "cosmoshub-4", accountNumber, sequence);
byte[] digest = CryptoNative.Sha256(CosmosTxBuilder.EncodeSignDoc(builder.EncodeBody(), builder.EncodeAuthInfo(account.PublicKey.Span, sequence), "cosmoshub-4", accountNumber));
Console.WriteLine($"{txRaw.Length} bytes, tx hash {CosmosTxBuilder.Hash(txRaw)}");
Console.WriteLine($"signature verifies: {CryptoNative.Secp256k1Verify(account.PublicKey.Span, digest, txRaw[^64..])}");

## Gas simulation by a real node

The node decodes and executes the transaction without committing it. A result proves the protobuf encoding is correct. Nothing is broadcast.

In [ ]:
try
{
    ulong gas = await rest.SimulateAsync(txRaw);
    Console.WriteLine($"gas used {gas:N0} → a safe limit is {(ulong)(gas * 1.3):N0}");
}
catch (RpcException e) { Console.WriteLine($"node rejected the simulation: {e.RpcMessage}"); }

## Sending for real

`SignAndBroadcastAsync` simulates, sizes gas (×1.3), pays the chain's gas price, signs and broadcasts:

```csharp
var testnet = wallet.GetCosmosAccount(0, CosmosChain.CosmosHubTestnet);
await using var client = new CosmosRestClient(CosmosChain.CosmosHubTestnet);
TxHash hash = await client.TransferAsync(testnet, "cosmos1…", Amount.FromAtom(0.1m), memo: "thanks");
```

In [ ]:
account.Dispose(); rest.Dispose(); wallet.Dispose();
"disposed"